In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""

处理步骤：
  1) 扫描 XML，找出含 ROI 的样本（按文件名前8位编号匹配）
  2) 仅处理这些样本对应的 DICOM
  3) DICOM 归一化到 [0,255]，保存 JPG（可选CLAHE/Resize）
  4) 由 XML 的 Point_px 多边形生成 mask，保存 PNG(0/255)
  5) 随机 8/1/1 划分到 train/val/test
"""

import os, re, plistlib, cv2, csv, random
import numpy as np
import pydicom
from pathlib import Path
from tqdm import tqdm
import pandas as pd


# path
ROOT = Path(r"D:\python_code\proj\INbreast Release 1.0")
OUT_ROOT = ROOT / "dataset_filtered"

# preprocess
APPLY_CLAHE = True         # 是否对 JPG 做自适应直方图均衡（对乳腺片常有帮助）
CLAHE_CLIP = 3.0
CLAHE_TILE = (8, 8)

RESIZE_TO = None           
RANDOM_SEED = 2025         
SPLIT = (0.8, 0.1, 0.1)    # train/val/test


def parse_point_px(str_list):
    """解析 ['(x, y)', '(x, y)', ...] 为 (N,2) float 数组"""
    pts = []
    for s in str_list:
        nums = [float(t) for t in re.findall(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", s)]
        if len(nums) >= 2:
            pts.append([nums[0], nums[1]])
    return np.array(pts, dtype=float)


def xml_has_roi(xml_path):
    """判断 INbreast 的 plist-xml 是否包含至少一个 ROI 多边形点集"""
    try:
        with open(xml_path, "rb") as f:
            pl = plistlib.load(f)
    except Exception:
        return False

    for img in pl.get("Images", []):
        for r in img.get("ROIs", []):
            pts = parse_point_px(r.get("Point_px", []))
            if len(pts) > 2:
                return True
    return False


def xml_to_mask(xml_path, img_shape):
    """从 INbreast XML 文件生成二值 mask（0/255）"""
    mask = np.zeros(img_shape[:2], dtype=np.uint8)
    try:
        with open(xml_path, "rb") as f:
            pl = plistlib.load(f)
    except Exception as e:
        print(f"[跳过] 无法解析 {xml_path.name}: {e}")
        return mask

    
    for img in pl.get("Images", []):
        for r in img.get("ROIs", []):
            pts = parse_point_px(r.get("Point_px", []))
            if len(pts) > 2:
                pts = np.round(pts).astype(np.int32)
                cv2.fillPoly(mask, [pts], 255)
    return mask


def load_and_preprocess_dicom(dcm_path):
    """读取 DICOM 并做归一化 [0,255]，返回 uint8 灰度图"""
    dcm = pydicom.dcmread(dcm_path)
    img = dcm.pixel_array.astype(np.float32)

    # 防止全零或异常情况
    imin, imax = float(img.min()), float(img.max())
    if imax - imin < 1e-8:
        img = np.zeros_like(img, dtype=np.uint8)
    else:
        img = (img - imin) / (imax - imin)
        img = (img * 255).astype(np.uint8)

    # 调整尺寸
    if RESIZE_TO is not None:
        img = cv2.resize(img, (RESIZE_TO[1], RESIZE_TO[0]), interpolation=cv2.INTER_AREA)

    return img


def maybe_resize_mask(mask):
    if RESIZE_TO is not None and mask.shape[:2] != RESIZE_TO:
        mask = cv2.resize(mask, (RESIZE_TO[1], RESIZE_TO[0]), interpolation=cv2.INTER_NEAREST)
    return mask


def main():
    random.seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)

    dcm_dir = ROOT / "AllDICOMs"
    xml_dir = ROOT / "AllXML"

    (OUT_ROOT / "images" / "train").mkdir(parents=True, exist_ok=True)
    (OUT_ROOT / "images" / "val").mkdir(parents=True, exist_ok=True)
    (OUT_ROOT / "images" / "test").mkdir(parents=True, exist_ok=True)
    (OUT_ROOT / "labels" / "train").mkdir(parents=True, exist_ok=True)
    (OUT_ROOT / "labels" / "val").mkdir(parents=True, exist_ok=True)
    (OUT_ROOT / "labels" / "test").mkdir(parents=True, exist_ok=True)
    (OUT_ROOT / "meta").mkdir(parents=True, exist_ok=True)

    # 1) 筛出“有 ROI 的 XML”
    xml_files = list(xml_dir.glob("*.xml"))
    xml_map = {x.stem[:8]: x for x in xml_files}  
    has_roi_prefix = set()

    print("扫描 XML（寻找含 ROI 的标注）...")
    for x in tqdm(xml_files, desc="Scanning XML"):
        if xml_has_roi(x):
            has_roi_prefix.add(x.stem[:8])

    print(f"共发现 XML 文件：{len(xml_files)}，其中含 ROI 的：{len(has_roi_prefix)}")

    # 2) 遍历 DICOM，仅处理“前8位在 has_roi_prefix” 的样本
    dcm_files = list(dcm_dir.glob("*.dcm"))
    print(f"找到 DICOM 文件：{len(dcm_files)}")

    # 记录所有“可用样本”（有 ROI 的）
    usable = []
    for dcm_path in tqdm(dcm_files, desc="Indexing DICOMs"):
        base = dcm_path.stem
        prefix = base[:8]
        if prefix in has_roi_prefix and prefix in xml_map:
            usable.append((dcm_path, xml_map[prefix]))

    print(f"将要处理的样本（有分割标签）：{len(usable)}")

    # 3) 划分 train/val/test
    random.shuffle(usable)
    n = len(usable)
    n_train = int(n * SPLIT[0])
    n_val = int(n * SPLIT[1])
    n_test = n - n_train - n_val
    splits = {
        "train": usable[:n_train],
        "val": usable[n_train:n_train + n_val],
        "test": usable[n_train + n_val:],
    }
    print(f"划分：train={len(splits['train'])}, val={len(splits['val'])}, test={len(splits['test'])}")

    # 4) 逐个读取 DICOM、生成 JPG 与 mask，并保存到对应 split
    meta_rows = []
    for split_name, items in splits.items():
        for dcm_path, xml_path in tqdm(items, desc=f"Processing {split_name}"):
            base = dcm_path.stem

            # 读取并预处理 DICOM → JPG
            try:
                img = load_and_preprocess_dicom(dcm_path)
            except Exception as e:
                print(f"[跳过] 无法读取 DICOM {dcm_path.name}: {e}")
                continue

            # 生成 mask
            try:
                mask = xml_to_mask(xml_path, img.shape)
            except Exception as e:
                print(f"[跳过] 无法生成 mask（{xml_path.name}）: {e}")
                continue

            # 若做了 resize，mask 也需相同尺寸
            mask = maybe_resize_mask(mask)

            # 保存
            jpg_path = OUT_ROOT / "images" / split_name / f"{base}.jpg"
            msk_path = OUT_ROOT / "labels" / split_name / f"{base}_mask.png"
            cv2.imwrite(str(jpg_path), img)
            cv2.imwrite(str(msk_path), mask)

            # 记录元信息
            meta_rows.append({
                "split": split_name,
                "id": base,
                "dcm_path": str(dcm_path),
                "xml_path": str(xml_path),
                "jpg_path": str(jpg_path),
                "mask_path": str(msk_path),
                "height": img.shape[0],
                "width": img.shape[1],
                "resize_to": str(RESIZE_TO) if RESIZE_TO is not None else "None",
                "clahe": int(APPLY_CLAHE)
            })

    # 5) 保存 meta 明细
    meta_csv = OUT_ROOT / "meta" / "inbreast_filtered_meta.csv"
    if meta_rows:
        pd.DataFrame(meta_rows).to_csv(meta_csv, index=False, encoding="utf-8-sig")
        print(f"\n处理完成，明细已保存：{meta_csv}")
    else:
        print("0")

    print("\n输出目录：")
    print(f"images/train: {OUT_ROOT / 'images' / 'train'}")
    print(f"images/val  : {OUT_ROOT / 'images' / 'val'}")
    print(f"images/test : {OUT_ROOT / 'images' / 'test'}")
    print(f"labels/*    : {OUT_ROOT / 'labels'}")


if __name__ == "__main__":
    main()


扫描 XML（寻找含 ROI 的标注）...


Scanning XML: 100%|██████████| 343/343 [00:00<00:00, 735.33it/s]


共发现 XML 文件：343，其中含 ROI 的：300
找到 DICOM 文件：410


Indexing DICOMs: 100%|██████████| 410/410 [00:00<00:00, 518596.09it/s]


将要处理的样本（有分割标签）：300
划分：train=240, val=30, test=30


Processing test: 100%|██████████| 30/30 [00:02<00:00, 13.25it/s]


✅ 处理完成，明细已保存：D:\python_code\proj\INbreast Release 1.0\dataset_filtered\meta\inbreast_filtered_meta.csv

输出目录：
images/train: D:\python_code\proj\INbreast Release 1.0\dataset_filtered\images\train
images/val  : D:\python_code\proj\INbreast Release 1.0\dataset_filtered\images\val
images/test : D:\python_code\proj\INbreast Release 1.0\dataset_filtered\images\test
labels/*    : D:\python_code\proj\INbreast Release 1.0\dataset_filtered\labels


In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

import cv2, random
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import albumentations as A

# ====== 路径设置（与上一步保持一致）======
ROOT = Path(r"D:\python_code\proj\INbreast Release 1.0")
IN_ROOT = ROOT / "dataset_filtered"        # 源：dataset_filtered/images/<split>、labels/<split>
OUT_ROOT = ROOT / "dataset_filtered_aug"   # 目标：dataset_filtered_aug/images/<split>、labels/<split>

SPLITS = ["train", "val", "test"]
IMG_DIRNAME = "images"
MSK_DIRNAME = "labels"

AUG_TIMES = 3
RANDOM_SEED = 2025
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ====== Pipelines ======
# T0 = A.Compose([
#     A.HorizontalFlip(p=0.5),
#     A.ShiftScaleRotate(shift_limit=0.02, scale_limit=0.10, rotate_limit=10,
#                        interpolation=cv2.INTER_LINEAR, border_mode=cv2.BORDER_REFLECT_101, p=0.9),
#     A.RandomBrightnessContrast(0.1, 0.1, p=0.7),
#     A.GaussNoise(var_limit=(5.0, 20.0), p=0.4),
# ])
# T1 = A.Compose([
#     A.VerticalFlip(p=0.4),
#     A.Affine(scale=(0.95, 1.05), translate_percent=(0.0, 0.02), rotate=(-7, 7),
#              shear=(-5, 5), fit_output=False, cval=0, p=0.9),
#     A.CLAHE(clip_limit=(2, 3), tile_grid_size=(8, 8), p=0.5),
#     A.GaussianBlur(blur_limit=(3,5), p=0.3),
# ])
# T2 = A.Compose([
#     A.ElasticTransform(alpha=10, sigma=10, alpha_affine=5,
#                        interpolation=cv2.INTER_LINEAR, border_mode=cv2.BORDER_REFLECT_101, p=0.5),
#     A.GridDistortion(num_steps=4, distort_limit=0.1, p=0.4),
#     A.PiecewiseAffine(scale=(0.01, 0.03), p=0.4),
#     A.RandomGamma(gamma_limit=(85, 115), p=0.5),
# ])

# 新增四组（a3, a4, a5, a6）：纯几何变换
# a3: 水平向右 +5% 平移
# T3 = A.Compose([
#     A.Affine(scale=1.0,
#              translate_percent={"x": (0.05, 0.05), "y": (0.0, 0.0)},
#              rotate=0, shear=0,
#              interpolation=cv2.INTER_LINEAR,
#              cval=0, fit_output=False,
#              mode=cv2.BORDER_REFLECT_101, p=1.0),
# ])

# # a4: 水平向左 -5% 平移
# T4 = A.Compose([
#     A.Affine(scale=1.0,
#              translate_percent={"x": (-0.05, -0.05), "y": (0.0, 0.0)},
#              rotate=0, shear=0,
#              interpolation=cv2.INTER_LINEAR,
#              cval=0, fit_output=False,
#              mode=cv2.BORDER_REFLECT_101, p=1.0),
# ])

# # a5: 顺时针 +12° 旋转
# T5 = A.Compose([
#     A.Affine(scale=1.0,
#              translate_percent={"x": (0.0, 0.0), "y": (0.0, 0.0)},
#              rotate=(12, 12),  # 固定 +12°
#              shear=0,
#              interpolation=cv2.INTER_LINEAR,
#              cval=0, fit_output=False,
#              mode=cv2.BORDER_REFLECT_101, p=1.0),
# ])

T3 = A.Compose([
    A.Affine(
        scale=1.0,
        translate_percent={"x": (0.05, 0.05), "y": (0.0, 0.0)},
        rotate=0, shear=0,
        interpolation=cv2.INTER_LINEAR,          # 图像插值
        mask_interpolation=cv2.INTER_NEAREST,    # 掩膜最近邻，避免灰化
        mode=cv2.BORDER_CONSTANT,                # 边界常量填充
        cval=0,                                  # 图像边界填 0（黑）
        cval_mask=0,                             # 掩膜边界填 0（背景）
        fit_output=False, p=1.0
    ),
])

# a4: 水平向左 -5% 平移
T4 = A.Compose([
    A.Affine(
        scale=1.0,
        translate_percent={"x": (-0.05, -0.05), "y": (0.0, 0.0)},
        rotate=0, shear=0,
        interpolation=cv2.INTER_LINEAR,
        mask_interpolation=cv2.INTER_NEAREST,
        mode=cv2.BORDER_CONSTANT,
        cval=0, cval_mask=0,
        fit_output=False, p=1.0
    ),
])

# a5: 顺时针 +12° 旋转
T5 = A.Compose([
    A.Affine(
        scale=1.0,
        translate_percent={"x": (0.0, 0.0), "y": (0.0, 0.0)},
        rotate=(12, 12),  # 固定 +12°
        shear=0,
        interpolation=cv2.INTER_LINEAR,
        mask_interpolation=cv2.INTER_NEAREST,
        mode=cv2.BORDER_CONSTANT,
        cval=0, cval_mask=0,
        fit_output=False, p=1.0
    ),
])

# T6：旋转幅度加大（±10°）+ CLAHE（50% 概率）
T6 = A.Compose([
    A.Rotate(
        limit=10, p=1.0,
        border_mode=cv2.BORDER_CONSTANT,
        value=0, mask_value=0
    ),
    A.CLAHE(p=0.5),   # 新增 CLAHE
])

# T7：旋转（±5°）+ 必做水平翻转 + CLAHE（50% 概率）
T7 = A.Compose([
    A.Rotate(
        limit=5, p=1.0,
        border_mode=cv2.BORDER_CONSTANT,
        value=0, mask_value=0
    ),
    A.HorizontalFlip(p=1.0),
    A.CLAHE(p=0.5),   # 新增 CLAHE
])

# T8：旋转（±5°）+ 水平翻转(50%) + CLAHE(必做)
# 原本 CLAHE 是 50% 概率，现在改为必做，增加对比度变化的多样性
T8 = A.Compose([
    A.Rotate(
        limit=5, p=1.0,
        border_mode=cv2.BORDER_CONSTANT,
        value=0, mask_value=0
    ),
    A.HorizontalFlip(p=0.5),
    A.CLAHE(p=1.0),   # CLAHE 改为必做
])



PIPELINES = [ T3, T4, T5, T6, T7,T8]


# ====== 工具 ======
def read_gray(p): 
    img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
    if img is None: raise FileNotFoundError(p)
    return img

def ensure_same_hw(a,b):
    if a.shape != b.shape:
        raise ValueError(f"Image/Mask size mismatch: {a.shape} vs {b.shape}")

def list_pairs(in_img_root, in_msk_root, split):
    img_dir = in_img_root / split
    msk_dir = in_msk_root / split
    items = []
    for p in img_dir.glob("*.jpg"):
        m = msk_dir / f"{p.stem}_mask.png"
        if m.exists():
            items.append((p, m))
    return items

def main():
    in_img_root = IN_ROOT / IMG_DIRNAME
    in_msk_root = IN_ROOT / MSK_DIRNAME

    out_img_root = OUT_ROOT / IMG_DIRNAME
    out_msk_root = OUT_ROOT / MSK_DIRNAME
    for split in SPLITS:
        (out_img_root / split).mkdir(parents=True, exist_ok=True)
        (out_msk_root / split).mkdir(parents=True, exist_ok=True)

    meta_rows = []
    total_pairs = 0
    for split in SPLITS:
        pairs = list_pairs(in_img_root, in_msk_root, split)
        print(f"[{split}] pairs: {len(pairs)}")
        total_pairs += len(pairs)

        for img_path, msk_path in tqdm(pairs, desc=f"Aug {split}"):
            img = read_gray(img_path)
            msk = read_gray(msk_path)
            ensure_same_hw(img, msk)

            for i, T in enumerate(PIPELINES):
                out_img = out_img_root / split / f"{img_path.stem}_a{i}.jpg"
                out_msk = out_msk_root / split / f"{img_path.stem}_a{i}_mask.png"
                ##同步mask
                tr = T(image=img, mask=msk)
                img_aug, msk_aug = tr["image"], tr["mask"]
                ensure_same_hw(img_aug, msk_aug)

                cv2.imwrite(str(out_img), img_aug)
                cv2.imwrite(str(out_msk), msk_aug)

                meta_rows.append({
                    "split": split,
                    "src_image": str(img_path),
                    "src_mask": str(msk_path),
                    "aug_id": i,
                    "dst_image": str(out_img),
                    "dst_mask": str(out_msk),
                    "H": img.shape[0], "W": img.shape[1],
                })


    meta_csv = OUT_ROOT / "meta_aug.csv"
    pd.DataFrame(meta_rows).to_csv(meta_csv, index=False, encoding="utf-8-sig")
    print(f"\n Augmentation done. CSV: {meta_csv}")
    print(f"Output root: {OUT_ROOT}")
    for split in SPLITS:
        print(f"- {split} images out: {out_img_root/split}")
        print(f"- {split} masks  out: {out_msk_root/split}")

if __name__ == "__main__":
    main()


[train] pairs: 240


Aug train: 100%|██████████| 240/240 [00:56<00:00,  4.22it/s]


[val] pairs: 30


Aug val: 100%|██████████| 30/30 [00:07<00:00,  4.20it/s]


[test] pairs: 30


Aug test: 100%|██████████| 30/30 [00:06<00:00,  4.33it/s]


✅ Augmentation done. CSV: D:\python_code\proj\INbreast Release 1.0\dataset_filtered_aug\meta_aug.csv
Output root: D:\python_code\proj\INbreast Release 1.0\dataset_filtered_aug
- train images out: D:\python_code\proj\INbreast Release 1.0\dataset_filtered_aug\images\train
- train masks  out: D:\python_code\proj\INbreast Release 1.0\dataset_filtered_aug\labels\train
- val images out: D:\python_code\proj\INbreast Release 1.0\dataset_filtered_aug\images\val
- val masks  out: D:\python_code\proj\INbreast Release 1.0\dataset_filtered_aug\labels\val
- test images out: D:\python_code\proj\INbreast Release 1.0\dataset_filtered_aug\images\test
- test masks  out: D:\python_code\proj\INbreast Release 1.0\dataset_filtered_aug\labels\test


In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

import re
import shutil
from pathlib import Path
from tqdm import tqdm
from typing import Optional

ROOT = Path(r"D:\python_code\proj\INbreast Release 1.0")

SRC_FILTERED = ROOT / "dataset_filtered"        # images/<split>/*.jpg  +  labels/<split>/*_mask.png
SRC_AUG      = ROOT / "dataset_filtered_aug"    # images/<split>/*_a*.jpg + labels/<split>/*_a*_mask.png
DST_FINAL    = ROOT / "dataset_final"           # 输出：dataset_final/<split>/{images,labels}

SPLITS = ["train", "val", "test"]

def leading_digits(stem: str) -> str:
    m = re.match(r"^(\d+)", stem)
    return m.group(1) if m else stem

def aug_suffix(stem: str) -> Optional[str]:
    """匹配任意 _a<number>，返回 a<number>，否则 None（支持 a0..aN）"""
    m = re.search(r"_a(\d+)$", stem)   # ← 原来是 [0-2]，现在放宽为任意数字
    return f"a{m.group(1)}" if m else None

def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def copy_pair(src_img: Path, src_msk: Path, dst_img: Path, dst_msk: Path, stats: dict):
    if dst_img.exists() or dst_msk.exists():
        stats["skipped_exist"] += 1
        return
    if not src_img.exists() or not src_msk.exists():
        stats["skipped_missing"] += 1
        return
    shutil.copy2(src_img, dst_img)
    shutil.copy2(src_msk, dst_msk)
    stats["copied"] += 1

def main():
    ensure_dir(DST_FINAL)
    summary = {}

    for split in SPLITS:
        out_img_dir = DST_FINAL / split / "images"
        out_msk_dir = DST_FINAL / split / "labels"
        ensure_dir(out_img_dir)
        ensure_dir(out_msk_dir)

        stats = {"copied": 0, "skipped_exist": 0, "skipped_missing": 0}
        summary[split] = stats

        # 1) 合并原始
        src_img_dir = SRC_FILTERED / "images" / split
        src_msk_dir = SRC_FILTERED / "labels" / split
        img_list = sorted(src_img_dir.glob("*.jpg"))
        print(f"[{split}] merging originals: {len(img_list)} files")

        for img_path in tqdm(img_list, desc=f"orig {split}"):
            stem = img_path.stem
            idnum = leading_digits(stem)
            msk_path = src_msk_dir / f"{stem}_mask.png"
            dst_img = out_img_dir / f"{idnum}.jpg"
            dst_msk = out_msk_dir / f"{idnum}_mask.png"
            copy_pair(img_path, msk_path, dst_img, dst_msk, stats)

        # 2) 合并增强（a0..aN 均可）
        aug_img_dir = SRC_AUG / "images" / split
        aug_msk_dir = SRC_AUG / "labels" / split
        aug_list = sorted(aug_img_dir.glob("*.jpg"))
        print(f"[{split}] merging augmentations: {len(aug_list)} files")

        for img_path in tqdm(aug_list, desc=f"aug  {split}"):
            stem = img_path.stem
            idnum = leading_digits(stem)
            suf = aug_suffix(stem)               # a0/a1/.../aN
            if suf is None:
                continue
            msk_path = aug_msk_dir / f"{stem}_mask.png"
            dst_img = out_img_dir / f"{idnum}_{suf}.jpg"
            dst_msk = out_msk_dir / f"{idnum}_{suf}_mask.png"
            copy_pair(img_path, msk_path, dst_img, dst_msk, stats)

        print(f"[{split}] done. copied={stats['copied']}, "
              f"skip_exist={stats['skipped_exist']}, skip_missing={stats['skipped_missing']}")

    print("\n All splits merged into:", DST_FINAL)
    for s, st in summary.items():
        print(f"- {s}: copied={st['copied']}  skip_exist={st['skipped_exist']}  skip_missing={st['skipped_missing']}")

if __name__ == "__main__":
    main()



[train] merging originals: 240 files


orig train: 100%|██████████| 240/240 [00:00<00:00, 871.24it/s]


[train] merging augmentations: 1440 files


aug  train: 100%|██████████| 1440/1440 [00:12<00:00, 118.48it/s]


[train] done. copied=1680, skip_exist=0, skip_missing=0
[val] merging originals: 30 files


...

[val] done. copied=210, skip_exist=0, skip_missing=0
[test] merging originals: 30 files


orig test: 100%|██████████| 30/30 [00:00<00:00, 872.33it/s]


[test] merging augmentations: 180 files


aug  test: 100%|██████████| 180/180 [00:01<00:00, 127.76it/s]

[test] done. copied=210, skip_exist=0, skip_missing=0

✅ All splits merged into: D:\python_code\proj\INbreast Release 1.0\dataset_final
- train: copied=1680  skip_exist=0  skip_missing=0
- val: copied=210  skip_exist=0  skip_missing=0
- test: copied=210  skip_exist=0  skip_missing=0
